In [ ]:
%run "./00_CC_Mapping_Setup.ipynb"

In [ ]:
%sql
/* ===================================================================================
   METRIC: EBA06 - Donations
   
   BUSINESS QUESTION: 
   During the assessment period, did the Unit, either directly or through the use of 
   third parties, make any domestic/international charitable donations?
   
   LOGIC SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
      Only AUs in this filtered list (Canada, HK, Barbados + US_OR_CANADA = 'CANADA') 
      will appear in the final report.
   2. FINANCE DATA ONLY: Unions the 6 Finance files. Coupa data is entirely excluded 
      based on the business logic.
   3. CATEGORY CODE FILTER: Strictly filters the Finance data for Category Codes 
      '292' and '427' (Charitable Donations).
   4. MAPPING & OUTPUT: Maps flagged transactions to AU_IDs using the CC Bootstrap 
      (with safe Integer casting on cost_center_id). LEFT JOINS this mapped data back 
      to the Master AU anchor and returns donation transaction count.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Combined_Finance_Raw AS (
    SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_1
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_2
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_3
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_4
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_5
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_6
),

All_Finance_Transactions AS (
    SELECT 
        CASE WHEN TRIM(CostCenter) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(CostCenter), 4), 4, '0') ELSE UPPER(TRIM(CostCenter)) END AS Cleaned_CC,
        TRIM(CatCode) AS CatCode
    FROM Combined_Finance_Raw
    WHERE CostCenter IS NOT NULL
),

Donation_Transactions AS (
    SELECT Cleaned_CC, CatCode
    FROM All_Finance_Transactions
    WHERE CatCode IN ('292', '427')
),

CC_Mapping AS (
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        CASE WHEN TRIM(cost_center_id) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(cost_center_id), 4), 4, '0') ELSE UPPER(TRIM(cost_center_id)) END AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
),

Flagged_AUs AS (
    SELECT 
        m.AU_ID,
        COUNT(*) AS Flagged_Count
    FROM Donation_Transactions d
    INNER JOIN CC_Mapping m
        ON d.Cleaned_CC = m.Mapped_CC
    GROUP BY m.AU_ID
)

SELECT 
    a.BusinessID,
    a.AU_Name,
    a.Business_Segment,
    'EBA06' AS QuestionID,
    COALESCE(f.Flagged_Count, 0) AS Response,
    a.In_ABAC_AU_List
FROM Master_AUs a
LEFT JOIN Flagged_AUs f
    ON a.BusinessID = f.AU_ID
ORDER BY a.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: EBA06 - AU Level Calculation Review
   PURPOSE: One row per AU showing normalized Cost Centers and how the final count
   response was calculated.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Combined_Finance_Raw AS (
    SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_1
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_2
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_3
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_4
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_5
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_6
),

All_Finance_Transactions AS (
    SELECT 
        CASE WHEN TRIM(CostCenter) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(CostCenter), 4), 4, '0') ELSE UPPER(TRIM(CostCenter)) END AS Cleaned_CC,
        TRIM(CatCode) AS CatCode
    FROM Combined_Finance_Raw
    WHERE CostCenter IS NOT NULL
),

Donation_Transactions AS (
    SELECT Cleaned_CC, CatCode
    FROM All_Finance_Transactions
    WHERE CatCode IN ('292', '427')
),

CC_Mapping AS (
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        CASE WHEN TRIM(cost_center_id) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(cost_center_id), 4), 4, '0') ELSE UPPER(TRIM(cost_center_id)) END AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
),

Mapped_All_Finance AS (
    SELECT 
        m.AU_ID,
        f.Cleaned_CC,
        f.CatCode
    FROM All_Finance_Transactions f
    INNER JOIN CC_Mapping m
        ON f.Cleaned_CC = m.Mapped_CC
),

Mapped_Donations AS (
    SELECT 
        m.AU_ID,
        d.Cleaned_CC,
        d.CatCode
    FROM Donation_Transactions d
    INNER JOIN CC_Mapping m
        ON d.Cleaned_CC = m.Mapped_CC
),

Donation_By_AU AS (
    SELECT 
        AU_ID,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(Cleaned_CC))) AS Normalized_CC_List,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(CatCode))) AS Donation_CatCode_List,
        COUNT(*) AS Donation_Row_Count
    FROM Mapped_Donations
    GROUP BY AU_ID
),

Finance_Record_By_AU AS (
    SELECT 
        AU_ID,
        COUNT(*) AS Mapped_Row_Count
    FROM Mapped_All_Finance
    GROUP BY AU_ID
)

-- Output columns:
-- BusinessID: Master AU ID from the ABAC AU anchor list.
-- AU_Name: Master AU name tied to BusinessID.
-- Business_Segment: Business segment carried from the master AU list.
-- QuestionID: Questionnaire code for this debug output.
-- Response: Final donation-row count returned for EBA06.
-- Normalized_CC_List: Normalized cost centers mapped to this AU.
-- Donation_CatCode_List: Donation-related category codes found on counted rows.
-- Mapped_Row_Count: Total mapped finance rows available for this AU.
-- Donation_Row_Count: Rows counted as donations after the EBA06 category-code filter.
-- Final_Count: Duplicate of the final donation-row total used for troubleshooting.
-- Calculation_Formula: Plain-English explanation of how Response was produced.
-- In_ABAC_AU_List: Confirms the AU came from the standard ABAC AU list.
SELECT 
    mast.BusinessID,
    mast.AU_Name,
    mast.Business_Segment,
    'EBA06' AS QuestionID,
    COALESCE(d.Donation_Row_Count, 0) AS Response,
    COALESCE(d.Normalized_CC_List, 'n/a') AS Normalized_CC_List,
    COALESCE(d.Donation_CatCode_List, 'n/a') AS Donation_CatCode_List,
    COALESCE(rec.Mapped_Row_Count, 0) AS Mapped_Row_Count,
    COALESCE(d.Donation_Row_Count, 0) AS Donation_Row_Count,
    COALESCE(d.Donation_Row_Count, 0) AS Final_Count,
    CONCAT('Response = ', CAST(COALESCE(d.Donation_Row_Count, 0) AS STRING), ' donation rows with category codes 292 or 427. Total mapped finance rows=', CAST(COALESCE(rec.Mapped_Row_Count, 0) AS STRING), '.') AS Calculation_Formula,
    mast.In_ABAC_AU_List
FROM Master_AUs mast
LEFT JOIN Donation_By_AU d
    ON mast.BusinessID = d.AU_ID
LEFT JOIN Finance_Record_By_AU rec
    ON mast.BusinessID = rec.AU_ID
ORDER BY mast.BusinessID;
